# Taller 1 — Consumo Automatizado de APIs

**Asignatura:** MLY1101 — Machine Learning  
**Nombre del estudiante:** Oreste Oscar  
**Sección:** MLY1101_001V  
**Fecha:** 2026-09-25

## Pregunta u objetivo

> ¿Qué información puede recopilarse sobre personajes y criaturas de distintos universos de entretenimiento (videojuegos, series animadas y anime)?

Para responder a esta pregunta de forma exploratoria se seleccionaron tres fuentes que, cada una por separado, aportan un catálogo de personajes/criaturas de un tipo distinto de medio: videojuegos, series animadas y anime. No se responde la pregunta ni se cruzan los datasets; cada fuente se descarga y almacena de forma independiente.


## Consideraciones generales

- Debe utilizar **3 APIs diferentes** disponibles en: https://github.com/public-apis/public-apis
- Cada API debe aportar información relacionada con el mismo objetivo.
- Debe obtener **mínimo 200 registros por API**, salvo que la fuente disponga de menos registros en total.
- Cada API debe generar un archivo independiente en formato `.json`, `.xlsx`, `.csv` o `.txt`.
- **No realizar merge, join, concat ni cruces entre datasets.**
- El notebook debe poder ejecutarse nuevamente usando **Entorno de ejecución → Ejecutar todas**.


In [16]:
# Librerías base
import requests
import json
import time
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path('datasets')
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Carpeta de salida: {OUTPUT_DIR.resolve()}')


Carpeta de salida: /Users/cabezon/Desktop/ML/Taller1/datasets


# Fuente 1 — API 1

**Nombre de la API:** PokeAPI  
**Documentación:** https://pokeapi.co/docs/v2  
**Endpoint utilizado:** `GET https://pokeapi.co/api/v2/pokemon?limit=100&offset=0` (paginado por `offset`)  
**Descripción de los datos:** Listado de Pokémon (criaturas de un videojuego), con `id`, `name` y `url` de detalle de cada uno.  
**Relación con el objetivo:** Aporta el catálogo de criaturas/personajes correspondiente al universo de los videojuegos.


In [17]:
# CONFIGURACIÓN API 1 — PokeAPI
API1_URL = 'https://pokeapi.co/api/v2/pokemon'
API1_MIN_REGISTROS = 200
API1_LIMIT_POR_PAGINA = 100


### Consumo y paginación — API 1

PokeAPI pagina mediante los parámetros `limit` y `offset`, y entrega en cada respuesta un campo `next` con la URL de la página siguiente (o `null` si no hay más). El ciclo avanza el `offset` hasta reunir al menos 200 registros o hasta que `next` sea `null`.


In [18]:
# CONSUMO Y PAGINACIÓN API 1 — PokeAPI
api1_registros = []
offset = 0

while len(api1_registros) < API1_MIN_REGISTROS:
    params = {'limit': API1_LIMIT_POR_PAGINA, 'offset': offset}
    response = requests.get(API1_URL, params=params, timeout=30)
    print('Status API 1, offset', offset, ':', response.status_code)
    response.raise_for_status()
    data = response.json()

    resultados = data.get('results', [])
    if not resultados:
        break

    for item in resultados:
        pokemon_id = item['url'].rstrip('/').split('/')[-1]
        api1_registros.append({
            'id': pokemon_id,
            'name': item['name'],
            'url': item['url']
        })

    if data.get('next') is None:
        break
    offset += API1_LIMIT_POR_PAGINA

print('Registros API 1 (PokeAPI):', len(api1_registros))


Status API 1, offset 0 : 200
Status API 1, offset 100 : 200
Registros API 1 (PokeAPI): 200


In [19]:
# GUARDAR DATASET API 1
API1_ARCHIVO = OUTPUT_DIR / 'dataset_api_1.csv'

df_api1 = pd.DataFrame(api1_registros)
df_api1.to_csv(API1_ARCHIVO, index=False, encoding='utf-8')

print('Archivo generado:', API1_ARCHIVO)
print('Registros guardados:', len(api1_registros))


Archivo generado: datasets/dataset_api_1.csv
Registros guardados: 200


# Fuente 2 — API 2

**Nombre de la API:** Rick and Morty API  
**Documentación:** https://rickandmortyapi.com/documentation  
**Endpoint utilizado:** `GET https://rickandmortyapi.com/api/character` (paginado siguiendo `info.next`)  
**Descripción de los datos:** Personajes de la serie animada Rick and Morty, con estado, especie, género, origen, ubicación e imagen.  
**Relación con el objetivo:** Aporta el catálogo de personajes correspondiente al universo de las series animadas.


In [20]:
# CONFIGURACIÓN API 2 — Rick and Morty API
API2_URL = 'https://rickandmortyapi.com/api/character'
API2_MIN_REGISTROS = 200


### Consumo y paginación — API 2

Cada respuesta incluye `info.next` con la URL exacta de la siguiente página (o `null` si es la última). El ciclo sigue ese enlace hasta reunir al menos 200 registros o hasta que no haya más páginas.


In [21]:
# CONSUMO Y PAGINACIÓN API 2 — Rick and Morty API
api2_registros = []
url_actual = API2_URL

while url_actual and len(api2_registros) < API2_MIN_REGISTROS:
    response = requests.get(url_actual, timeout=30)
    print('Status API 2:', response.status_code, '-', url_actual)
    response.raise_for_status()
    data = response.json()

    for personaje in data.get('results', []):
        api2_registros.append({
            'id': personaje.get('id'),
            'name': personaje.get('name'),
            'status': personaje.get('status'),
            'species': personaje.get('species'),
            'type': personaje.get('type'),
            'gender': personaje.get('gender'),
            'origin': (personaje.get('origin') or {}).get('name'),
            'location': (personaje.get('location') or {}).get('name'),
            'image': personaje.get('image'),
            'episode_count': len(personaje.get('episode', [])),
            'created': personaje.get('created')
        })

    url_actual = (data.get('info') or {}).get('next')

print('Registros API 2 (Rick and Morty):', len(api2_registros))


Status API 2: 200 - https://rickandmortyapi.com/api/character
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=2
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=3
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=4
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=5
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=6
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=7
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=8
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=9
Status API 2: 200 - https://rickandmortyapi.com/api/character?page=10
Registros API 2 (Rick and Morty): 200


In [22]:
# GUARDAR DATASET API 2
API2_ARCHIVO = OUTPUT_DIR / 'dataset_api_2.csv'

df_api2 = pd.DataFrame(api2_registros)
df_api2.to_csv(API2_ARCHIVO, index=False, encoding='utf-8')

print('Archivo generado:', API2_ARCHIVO)
print('Registros guardados:', len(api2_registros))


Archivo generado: datasets/dataset_api_2.csv
Registros guardados: 200


# Fuente 3 — API 3

**Nombre de la API:** Kitsu API  
**Documentación:** https://kitsu.docs.apiary.io/  
**Endpoint utilizado:** `GET https://kitsu.io/api/edge/characters?page[limit]=20&page[offset]=0` (paginado por `page[offset]`, siguiendo `links.next`)  
**Descripción de los datos:** Catálogo de personajes de anime/manga registrados en Kitsu, con nombre, nombre canónico, id de MyAnimeList asociado (si existe) y una breve descripción.  
**Relación con el objetivo:** Aporta el catálogo de personajes correspondiente al universo del anime.


In [23]:
# CONFIGURACIÓN API 3 — Kitsu API
API3_URL = 'https://kitsu.io/api/edge/characters'
API3_MIN_REGISTROS = 200
API3_LIMIT_POR_PAGINA = 20  # máximo permitido por página en Kitsu
API3_PAUSA_ENTRE_LLAMADAS = 0.3
API3_MAX_REINTENTOS = 5


### Consumo y paginación — API 3

Kitsu pagina mediante `page[limit]` y `page[offset]`, y entrega un objeto `links` con la clave `next` mientras existan más páginas. Cada página se solicita con un pequeño reintento con espera creciente por si hay un error transitorio de red o del servidor.


In [24]:
# CONSUMO Y PAGINACIÓN API 3 — Kitsu API
def obtener_pagina_api3(offset, intentos=API3_MAX_REINTENTOS):
    params = {'page[limit]': API3_LIMIT_POR_PAGINA, 'page[offset]': offset}
    for intento in range(1, intentos + 1):
        try:
            response = requests.get(API3_URL, params=params, timeout=30)
        except requests.exceptions.RequestException as error:
            print(f'  Intento {intento}/{intentos} falló (excepción de red): {error}')
        else:
            print('Status API 3, offset', offset, ', intento', intento, ':', response.status_code)
            if response.status_code == 200:
                return response.json()
            if response.status_code not in (429, 500, 502, 503, 504):
                response.raise_for_status()

        if intento < intentos:
            espera = API3_PAUSA_ENTRE_LLAMADAS * (2 ** (intento - 1))
            print(f'  Reintentando en {espera:.1f}s...')
            time.sleep(espera)

    raise RuntimeError(f'No se pudo obtener la página con offset {offset} de la API 3 tras {intentos} intentos.')

api3_registros = []
offset = 0

while len(api3_registros) < API3_MIN_REGISTROS:
    data = obtener_pagina_api3(offset)

    items = data.get('data', [])
    if not items:
        break

    for item in items:
        attrs = item.get('attributes') or {}
        api3_registros.append({
            'id': item.get('id'),
            'name': attrs.get('name'),
            'canonical_name': attrs.get('canonicalName'),
            'mal_id': attrs.get('malId'),
            'description': (attrs.get('description') or '')[:300]
        })

    if 'next' not in (data.get('links') or {}):
        break

    offset += API3_LIMIT_POR_PAGINA
    time.sleep(API3_PAUSA_ENTRE_LLAMADAS)

print('Registros API 3 (Kitsu - Personajes de anime):', len(api3_registros))


Status API 3, offset 0 , intento 1 : 200
Status API 3, offset 20 , intento 1 : 200
Status API 3, offset 40 , intento 1 : 200
Status API 3, offset 60 , intento 1 : 200
Status API 3, offset 80 , intento 1 : 200
Status API 3, offset 100 , intento 1 : 200
Status API 3, offset 120 , intento 1 : 200
Status API 3, offset 140 , intento 1 : 200
Status API 3, offset 160 , intento 1 : 200
Status API 3, offset 180 , intento 1 : 200
Registros API 3 (Kitsu - Personajes de anime): 200


In [25]:
# GUARDAR DATASET API 3
API3_ARCHIVO = OUTPUT_DIR / 'dataset_api_3.csv'

df_api3 = pd.DataFrame(api3_registros)
df_api3.to_csv(API3_ARCHIVO, index=False, encoding='utf-8')

print('Archivo generado:', API3_ARCHIVO)
print('Registros guardados:', len(api3_registros))


Archivo generado: datasets/dataset_api_3.csv
Registros guardados: 200


# Resumen final

La siguiente celda debe ejecutarse al final y mostrar el resultado real de la carga.


In [26]:
total = len(api1_registros) + len(api2_registros) + len(api3_registros)

print('RESUMEN DE CARGA')
print('-' * 50)
print(f'API 1 (PokeAPI): {len(api1_registros)} registros - {API1_ARCHIVO.name}')
print(f'API 2 (Rick and Morty): {len(api2_registros)} registros - {API2_ARCHIVO.name}')
print(f'API 3 (Kitsu - Anime): {len(api3_registros)} registros - {API3_ARCHIVO.name}')
print('-' * 50)
print(f'TOTAL: {total} registros')

if len(api1_registros) < 200:
    print('ADVERTENCIA: API 1 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api2_registros) < 200:
    print('ADVERTENCIA: API 2 tiene menos de 200 registros. Documente la excepción si corresponde.')
if len(api3_registros) < 200:
    print('ADVERTENCIA: API 3 tiene menos de 200 registros. Documente la excepción si corresponde.')


RESUMEN DE CARGA
--------------------------------------------------
API 1 (PokeAPI): 200 registros - dataset_api_1.csv
API 2 (Rick and Morty): 200 registros - dataset_api_2.csv
API 3 (Kitsu - Anime): 200 registros - dataset_api_3.csv
--------------------------------------------------
TOTAL: 600 registros


# Entrega en GitHub

El repositorio debe contener como mínimo:

```text
MLY1101-Taller1-OresteOscar/
├── MLY1101_001V_T01_OresteOscar.ipynb
├── dataset_api_1.csv
├── dataset_api_2.csv
├── dataset_api_3.csv
└── README.md
```

Si el repositorio es privado, agregar como colaborador a **titiriemann**.


## Checklist final

- [x] Definí una pregunta u objetivo común.
- [x] Utilicé 3 APIs del repositorio indicado (PokeAPI, Rick and Morty API, Jikan API).
- [x] Obtengo al menos 200 registros por API mediante paginación automática.
- [x] Genero 3 archivos independientes (.csv).
- [x] No uno ni cruzo los datasets.
- [x] El notebook ejecuta desde cero (Entorno de ejecución → Ejecutar todas).
- [ ] Subí notebook, datasets y README.md a GitHub.
- [ ] Entregué la URL del repositorio.
